[<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/> 2. Help Assistant](https://colab.research.google.com/github/AlbertoLopezCorbalan/MIA-TFM/blob/main/TFM_help_assistant.ipynb)  

# Trabajo Fin de Máster - Help Assistant
> Autor: Alberto López Corbalán alberto.lopezc@um.es

Tras el análisis de las primeras pruebas de predicción sobre si un nivel ha sido completado o no, se plantea un problema más complejo. El nuevo objetivo es predecir si un jugador necesita ayuda para completar un nivel, considerando si finalmente logrará completarlo o no.

Para ello, se propone utilizar la traza generada mientras el jugador interactúa con el juego (texto plano), procesándola mediante un modelo de lenguaje de gran tamaño (LLM). Con el fin de capturar un contexto más amplio, se empleará un modelo destilado de larga secuencia, específicamente `allenai/longformer-base-4096`.

El entrenamiento se realizará durante una única época, evaluando el modelo con la traza al 25%, al 50% y finalmente al 75% del total de la interacción del jugador, simulando distintos estados intermedios de la partida para analizar la capacidad predictiva del modelo en diferentes fases del juego.

Este notebook requiere un entorno con GPU para poder ejecutarse.


In [1]:
import sys

if "google.colab" in sys.modules:
    print("Google Colab")
    # Para obtener el dataset si se ejecuta en Colab
    !git clone https://github.com/AlbertoLopezCorbalan/MIA-TFM
    %cd MIA-TFM

Google Colab
Cloning into 'MIA-TFM'...
remote: Enumerating objects: 43, done.
remote: Counting objects: 100% (43/43), done.
remote: Compressing objects: 100% (29/29), done.
remote: Total 43 (delta 21), reused 31 (delta 12), pack-reused 0 (from 0)
Receiving objects: 100% (43/43), 3.96 MiB | 2.85 MiB/s, done.
Resolving deltas: 100% (21/21), done.
/content/MIA-TFM


In [2]:
import pandas as pd
import re
import numpy as np
import json
import matplotlib.pyplot as plt
import time
import torch
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split, GridSearchCV

# https://huggingface.co/allenai/longformer-base-4096 -> https://arxiv.org/pdf/2004.05150
from datasets import Dataset
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, TrainingArguments, Trainer
from transformers import LongformerTokenizerFast, LongformerForSequenceClassification

print(torch.cuda.get_device_name(0))

GLOBAL_SEED = 123 # Seed global utilizada para garantizar la reproducibilidad
DATASET_PATH = "dataset/complete_report_features_and_text_replay.csv"

Tesla T4


Para este problema solo será necesaria la traza y variable objetivo del dataset.


In [3]:
dataset = pd.read_csv(DATASET_PATH)
print(f"{DATASET_PATH}: " + str(len(dataset)))
dataset.head()

dataset/complete_report_features_and_text_replay.csv: 6995


,replay,user,group,puzzle,globalAttemptId,attemptId,contextFeatures,attemptFeatures,textReplay
0,2. Separated Boxes~1,2d3db94690a19a62a0942fbd6ac30308,4fe25833f555e9903d2bb6bbeec3fbfb,2. Separated Boxes,2,1,"{""#Attempt"": 1, ""#GlobalAttempt"": 2, ""ActiveTi...","{""ActiveTime"": 27.355294999999998, ""InactiveTi...",Player 2d3db94690a19a62a0942fbd6ac30308 starte...
1,3. Rotate a Pyramid~1,2d3db94690a19a62a0942fbd6ac30308,4fe25833f555e9903d2bb6bbeec3fbfb,3. Rotate a Pyramid,3,1,"{""#Attempt"": 1, ""#GlobalAttempt"": 3, ""ActiveTi...","{""ActiveTime"": 164.38663100000002, ""InactiveTi...",Player 2d3db94690a19a62a0942fbd6ac30308 starte...
2,3. Rotate a Pyramid~2,2d3db94690a19a62a0942fbd6ac30308,4fe25833f555e9903d2bb6bbeec3fbfb,3. Rotate a Pyramid,5,2,"{""#Attempt"": 2, ""#GlobalAttempt"": 5, ""ActiveTi...","{""ActiveTime"": 170.76343800000006, ""InactiveTi...",Player 2d3db94690a19a62a0942fbd6ac30308 starte...
3,Bear Market~1,2d3db94690a19a62a0942fbd6ac30308,4fe25833f555e9903d2bb6bbeec3fbfb,Bear Market,4,1,"{""#Attempt"": 1, ""#GlobalAttempt"": 4, ""ActiveTi...","{""ActiveTime"": 40.556214, ""InactiveTime"": 0, ""...",Player 2d3db94690a19a62a0942fbd6ac30308 starte...
4,4. Match Silhouettes~1,2d3db94690a19a62a0942fbd6ac30308,4fe25833f555e9903d2bb6bbeec3fbfb,4. Match Silhouettes,6,1,"{""#Attempt"": 1, ""#GlobalAttempt"": 6, ""ActiveTi...","{""ActiveTime"": 100.48931000000002, ""InactiveTi...",Player 2d3db94690a19a62a0942fbd6ac30308 starte...


# Preprocesamiento (25%/50%/75%)

A continuación, preparamos los datos acordes al experimento. Primero, parseamos los JSON contenidos en la columna attemptFeatures para extraer la variable objetivo `attempt_Completed`. Luego, seleccionamos únicamente la columna de texto `textReplay` junto con la variable objetivo. Finalmente, generamos tres versiones del DataFrame con diferentes fracciones del texto (25%, 50% y 75%) para entrenar y evaluar el modelo en distintas etapas de la interacción del jugador.

In [4]:
df_llm = dataset.copy(deep = True)

# Parseamos el json
df_llm["attemptFeatures"] = df_llm["attemptFeatures"].apply(json.loads)

# Normalizamos los json para pasarlos a una tabla aplanada
df_attempt = pd.json_normalize(df_llm["attemptFeatures"])

# Quitamos los "." por "_"
df_attempt.columns = df_attempt.columns.str.replace(".", "_", regex=False)

# Unimos todo
df_llm = pd.concat(
    [
        df_llm.drop(columns=["contextFeatures", "globalAttemptId", "attemptFeatures"]),
        df_attempt.add_prefix("attempt_"),
    ],
    axis=1
)

# Seleccionamos solo la textReplay y la variable objetivo
df_llm = df_llm[["textReplay", "attempt_Completed"]]
df_llm["attempt_Completed"] = df_llm["attempt_Completed"].astype("int64")

# Procedemos a modificar la columna textReplay para eliminar el texto inicial (información contextual) antes del primer evento `ws-start_level`
def remove_initial_text_until_start_level(text):
    # Buscar la primera aparición de 'ws-start_level'
    match = re.search(r'ws-start_level', text)

    if match:
        start_index = match.start()

        # Buscar el primer ' -> ' después de esa aparición
        arrow_index = text.find(' -> ', start_index)

        if arrow_index != -1:
            return text[arrow_index + len(' -> '):].strip()
        else:
            return text[match.end():].strip()

    return text

# Aplicar la función de preprocesamiento a la columna 'textReplay'
df_llm['textReplay'] = df_llm['textReplay'].apply(remove_initial_text_until_start_level)

# DataFrame al 25%
df_llm_25 = df_llm.copy(deep=True)
df_llm_25["textReplay"] = df_llm_25["textReplay"].apply(lambda x: x[:int(len(x) * 0.25)])

# DataFrame al 50%
df_llm_50 = df_llm.copy(deep=True)
df_llm_50["textReplay"] = df_llm_50["textReplay"].apply(lambda x: x[:int(len(x) * 0.5)])

# DataFrame al 75%
df_llm_75 = df_llm.copy(deep=True)
df_llm_75["textReplay"] = df_llm_75["textReplay"].apply(lambda x: x[:int(len(x) * 0.75)])

# Entrenamiento

El siguiente bloque define la función para entrenar el LLM. Primero, divide los datos en conjuntos de entrenamiento y prueba usando estratificación para mantener la proporción de clases. Luego, tokeniza los textos y los convierte en datasets compatibles con HuggingFace. Se carga el modelo *LongformerForSequenceClassification* con dos etiquetas y se configura un entrenamiento de una época. Durante el entrenamiento, se calcula el rendimiento usando métricas de precisión, recall y F1.

In [5]:
def train_llm_classifier(df, max_length=4096):

  # Parámetros de configuración
  model_name = "allenai/longformer-base-4096"
  epochs = 1
  train_batch_size = 2
  eval_batch_size = 2
  learning_rate = 2e-5
  weight_decay = 0.01


  # Divide el dataset en entrenamiento y test
  # usando estratificación para mantener proporción de clases
  x_train, x_test, y_train, y_test = train_test_split(df["textReplay"].tolist(), df["attempt_Completed"].tolist(),
                                                      test_size=0.2, random_state=GLOBAL_SEED, stratify=df["attempt_Completed"])

  # Carga el tokenizer del modelo
  tokenizer = LongformerTokenizerFast.from_pretrained(model_name)

   # Se tokeniza del conjunto de entrenamiento y prueba
  train_encodings = tokenizer(x_train, truncation=True, padding=True, max_length=max_length)
  test_encodings = tokenizer(x_test, truncation=True, padding=True, max_length=max_length)

  # Se convierte a formato Dataset de HuggingFace
  train_dataset = Dataset.from_dict({**train_encodings, "labels": y_train})
  test_dataset = Dataset.from_dict({**test_encodings, "labels": y_test})

  # Carga del modelo para clasificación
  model = LongformerForSequenceClassification.from_pretrained(model_name, num_labels=2)

  # Función para calcular rendimiento
  def compute_metrics(eval_pred):

      logits, labels = eval_pred

      # Obtiene la clase con mayor probabilidad
      predictions = np.argmax(logits, axis=-1)

      return {
          "f1": f1_score(labels, predictions),
          "precision": precision_score(labels, predictions),
          "recall": recall_score(labels, predictions),
      }

  # Configuración del entrenamiento
  training_args = TrainingArguments(
      eval_strategy = "epoch",
      save_strategy = "epoch",
      logging_strategy = "epoch",
      num_train_epochs = epochs,
      # Batch sizes
      per_device_train_batch_size = train_batch_size,
      per_device_eval_batch_size = eval_batch_size,
      learning_rate = learning_rate,
      weight_decay = weight_decay,
      load_best_model_at_end = True,
      fp16 = torch.cuda.is_available()
  )

  trainer = Trainer(
      model=model,
      args=training_args,
      train_dataset=train_dataset,
      eval_dataset=test_dataset,
      compute_metrics=compute_metrics,
  )

  trainer.train()

  # Mostramos el rendimiento
  metrics = trainer.evaluate()

  return metrics

En este bloque, entrenamos el clasificador utilizando la versión del DataFrame que contiene solo el 25% del texto de cada intento.

In [6]:
metrics_25 = train_llm_classifier(df_llm_25)
print(metrics_25)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/597M [00:00<?, ?B/s]

Epoch,Training Loss,Validation Loss,F1,Precision,Recall
1,0.786974,0.814777,0.827554,0.705835,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,F1,Precision,Recall
0.786974,0.814777,1,0.827554,0.705835,1.000000


{'eval_loss': 0.8147773742675781, 'eval_f1': 0.8275538894095595, 'eval_precision': 0.7058353317346123, 'eval_recall': 1.0}


Los resultados obtenidos utilizando únicamente el 25% de la traza muestran un buen comportamiento del modelo incluso con información parcial de la partida.

Destaca especialmente el recall, lo que significa que el modelo es capaz de identificar correctamente los casos positivos. Sin embargo, la precisión es menor, indicando la presencia de un número apreciable de falsos positivos.

In [7]:
metrics_50 = train_llm_classifier(df_llm_50)
print(metrics_50)

Loading weights:   0%|          | 0/269 [00:00<?, ?it/s]

[transformers] LongformerForSequenceClassification LOAD REPORT from: allenai/longformer-base-4096
Key                            | Status     | 
-------------------------------+------------+-
lm_head.decoder.weight         | UNEXPECTED | 
lm_head.layer_norm.bias        | UNEXPECTED | 
lm_head.layer_norm.weight      | UNEXPECTED | 
longformer.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias             | UNEXPECTED | 
lm_head.dense.weight           | UNEXPECTED | 
lm_head.bias                   | UNEXPECTED | 
longformer.pooler.dense.weight | UNEXPECTED | 
classifier.dense.bias          | MISSING    | 
classifier.out_proj.weight     | MISSING    | 
classifier.out_proj.bias       | MISSING    | 
classifier.dense.weight        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream ta

Epoch,Training Loss,Validation Loss,F1,Precision,Recall
1,0.758993,0.777114,0.822852,0.714167,0.970555


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,F1,Precision,Recall
0.758993,0.777114,1,0.822852,0.714167,0.970555


{'eval_loss': 0.7771139144897461, 'eval_f1': 0.8228516562650025, 'eval_precision': 0.7141666666666666, 'eval_recall': 0.970554926387316}


Los resultados obtenidos utilizando el 50% de la traza muestran que el modelo mantiene una elevada capacidad de detección, alcanzando un recall de `0.97`. Esto indica que prácticamente todos los casos positivos son identificados correctamente cuando se dispone de la mitad de la interacción del jugador.

In [8]:
metrics_75 = train_llm_classifier(df_llm_75)
print(metrics_75)

Loading weights:   0%|          | 0/269 [00:00<?, ?it/s]

[transformers] LongformerForSequenceClassification LOAD REPORT from: allenai/longformer-base-4096
Key                            | Status     | 
-------------------------------+------------+-
lm_head.decoder.weight         | UNEXPECTED | 
lm_head.layer_norm.bias        | UNEXPECTED | 
lm_head.layer_norm.weight      | UNEXPECTED | 
longformer.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias             | UNEXPECTED | 
lm_head.dense.weight           | UNEXPECTED | 
lm_head.bias                   | UNEXPECTED | 
longformer.pooler.dense.weight | UNEXPECTED | 
classifier.dense.bias          | MISSING    | 
classifier.out_proj.weight     | MISSING    | 
classifier.out_proj.bias       | MISSING    | 
classifier.dense.weight        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream ta

Epoch,Training Loss,Validation Loss,F1,Precision,Recall
1,0.841325,1.001592,0.825234,0.702466,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,F1,Precision,Recall
0.841325,1.001592,1,0.825234,0.702466,1.000000


{'eval_loss': 1.001591682434082, 'eval_f1': 0.8252336448598131, 'eval_precision': 0.7024661893396977, 'eval_recall': 1.0}


Los resultados obtenidos utilizando el 75% de la traza muestran valores similares a los observados en el escenario del 50%, indicando que el aumento de información no reduce significativamente la cantidad de falsos positivos generados por el modelo.

# Conclusiones

Los resultados obtenidos muestran que el modelo es capaz de realizar predicciones útiles incluso utilizando únicamente una fracción temprana de la interacción del jugador. Esto confirma que existen patrones relevantes en las primeras fases de la partida que permiten anticipar el comportamiento con una precisión razonable.

El mejor valor de F1 se obtiene utilizando el 25% de la traza, resultando ideal para nuestro contexto, donde se busca asistir al jugador durante el transcurso de la partida.